# DeepShield AI — Google Colab Model Training Notebook
### Deepfake Image and Video Detection System (ResNet-18 Transfer Learning)
**Student:** B.Tech AI & Data Science (2nd Year)  
**Goal:** Train deep learning binary classifier (REAL vs FAKE) using NVIDIA GPU acceleration in Colab, evaluate on held-out test data, and download `best_model.pth` for local Windows deployment.

## 1. Verify Hardware Acceleration (GPU)
Ensure your Google Colab runtime is set to **GPU** (`Runtime` -> `Change runtime type` -> `T4 GPU`).

In [ ]:
!nvidia-smi

import torch
print("PyTorch Version:", torch.__version__)
print("CUDA Available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU Device Name:", torch.cuda.get_device_name(0))
else:
    print("WARNING: No GPU detected! Go to Runtime -> Change runtime type -> T4 GPU.")

## 2. Install Required Dependencies

In [ ]:
!pip install -q pyyaml opencv-python pillow scikit-learn matplotlib seaborn tqdm pandas

## 3. Clone Repository or Set Up Directory Structure
If your project is hosted on GitHub, clone it here. Otherwise, the cell below creates the necessary directories.

In [ ]:
import os

for folder in ["configs", "data", "models", "outputs", "src", "scripts"]:
    os.makedirs(folder, exist_ok=True)

print("Colab working directories initialized!")

## 4. (Optional) Mount Google Drive
If your dataset is stored in Google Drive, mount it here. If uploading directly, skip this step.

In [ ]:
# from google.colab import drive
# drive.mount('/content/drive')
# # Example: unzip dataset from Drive
# !unzip -q /content/drive/MyDrive/deepfake_dataset.zip -d data/

## 5. Prepare or Verify Dataset
Check your dataset structure: `data/train/real`, `data/train/fake`, `data/val/real`, `data/val/fake`, `data/test/real`, `data/test/fake`.
If you want to run a quick pipeline test on Colab first, run `python scripts/prepare_dataset.py --generate_pipeline_dataset`.

In [ ]:
# Verify dataset contents
!python scripts/prepare_dataset.py --validate
# If data directory is empty, you can generate a pipeline test dataset using:
# !python scripts/prepare_dataset.py --generate_pipeline_dataset --samples 25

## 6. Train the Model (ResNet-18 Transfer Learning)
Runs full training loop on GPU, validating after every epoch and saving `models/best_model.pth`.

In [ ]:
# Execute training script on GPU with batch size 32
!python src/train.py --config configs/config.yaml --epochs 10 --batch_size 32 --device cuda

## 7. Display Training Convergence Curves

In [ ]:
from IPython.display import Image, display
import os

if os.path.exists('outputs/training_curves.png'):
    display(Image('outputs/training_curves.png'))
else:
    print("Training curves will be available once training completes.")

## 8. Evaluate on Held-Out Test Set
Computes Accuracy, Precision, Recall, F1-Score, Confusion Matrix, and ROC-AUC curve.

In [ ]:
!python src/evaluate.py --checkpoint models/best_model.pth --test_dir data/test

if os.path.exists('outputs/confusion_matrix.png'):
    display(Image('outputs/confusion_matrix.png'))
if os.path.exists('outputs/roc_curve.png'):
    display(Image('outputs/roc_curve.png'))

## 9. Download Trained Checkpoint to Local Windows PC
Execute the cell below to download `best_model.pth` directly to your computer.  
Place the downloaded file into your local project folder: `D:\DEEPFAKE_DETECTION\models\best_model.pth`.

In [ ]:
from google.colab import files
import os

if os.path.exists('models/best_model.pth'):
    print("Downloading best_model.pth...")
    files.download('models/best_model.pth')
else:
    print("Checkpoint file 'models/best_model.pth' not found.")